In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.preprocessing import StandardScaler

In [2]:
ruta = Path("../dvd_rental_muestra/csv")

actors = pd.read_csv(ruta / "actors.csv")
address = pd.read_csv(ruta / "address_cust.csv")
category = pd.read_csv(ruta / "category.csv")
customer = pd.read_csv(ruta / "customer.csv")
film = pd.read_csv(ruta / "film.csv")
film_actor = pd.read_csv(ruta / "film_actor.csv")
inventory = pd.read_csv(ruta / "inventory.csv")
language = pd.read_csv(ruta / "language.csv")
payment = pd.read_csv(ruta / "paymnet.csv")
rental = pd.read_csv(ruta / "rental.csv")

datos = {   #crear un diccionario nos ayudara a la limpieza de datos, para recorrer las tablas mediante un for sin repetir codigo
    "actors": actors,
    "address_cust": address,
    "category": category,
    "customer": customer,
    "film": film,
    "film_actor": film_actor,
    "inventory": inventory,
    "language": language,
    "payment": payment,
    "rental": rental
}


In [3]:
#2 limpieza de datos: en este caso se eliminaran telefono y codigo postal ya que no aportan nada 
# de infromacion,mientras que distrito se sustituira por desconocido, en adicion tienen infromacion personal de los clientes
limpios = {
    nombre: df.copy()
    for nombre, df in datos.items()
}
limpios["address_cust"]["district"] = (
    limpios["address_cust"]["district"].fillna("Desconocido")
)

limpios["address_cust"] = limpios["address_cust"].drop(
    columns=["address", "phone", "postal_code"]
)
    
print("Valores faltantes antes de la limpieza:")
print(datos["address_cust"].isnull().sum().to_string())

print("\nValores faltantes despues de la limpieza:")
print(limpios["address_cust"].isnull().sum().to_string())

Valores faltantes antes de la limpieza:
address_id     0
address        0
district       3
 city          0
postal_code    4
phone          2

Valores faltantes despues de la limpieza:
address_id    0
district      0
 city         0


In [10]:
#outliers, con esto podremos ver valores atipicos en las variables nuemricas de los csv
def detectar_iqr(serie):
    q1 = serie.quantile(0.25) #qn cuartiles
    q3 = serie.quantile(0.75)
    iqr = q3 - q1 #rango que contiene el 50% central de los datos.
#calculo de limites
    inferior = q1 - 1.5 * iqr
    superior = q3 + 1.5 * iqr
#checamos si un registro es atipico; si se aleja del resto del conjunto de datos
    mascara = (serie < inferior) | (serie > superior)

    return inferior, superior, mascara
#ejemplo: quiero encontrar valores atipicos en la colmuna de amount en la tabla de payment
inferior, superior, mascara = detectar_iqr(limpios["payment"]["amount"])

print("limite inferior:", inferior)
print("limite superior:", superior)
print("outliers:", mascara.sum())

print(limpios["payment"][mascara])
#en este caso tengo 2 outliers por que 2 registros tienen la misma cantidad(amount = 10.99)

###tratamiento de outliers
limpios["payment"]["outlier"] = mascara

print(limpios["payment"][mascara])

print(limpios["payment"]["outlier"].value_counts().to_string())


limite inferior: -1.5099999999999998
limite superior: 10.49
outliers: 3
     payment_id  customer_id  rental_id  amount payment_date  outlier
198       22323          565      10776   10.99   2017-03-01     True
250       24553          195      16040   11.99   2017-03-23     True
316       26471          380       9710   10.99   2017-04-30     True
     payment_id  customer_id  rental_id  amount payment_date  outlier
198       22323          565      10776   10.99   2017-03-01     True
250       24553          195      16040   11.99   2017-03-23     True
316       26471          380       9710   10.99   2017-04-30     True
outlier
False    497
True       3


In [11]:
#3 Normalizacion: se usara StandarScales pues algunas variables numericas tienen escalas y valores diferentes

normalizarFilm = [ #de la tabla film
    "length",
    "rental_duration",
    "rental_rate",
    "replacement_cost"
]
film_normalizado = limpios["film"].copy()

# Creamos el normalizador
scaler = StandardScaler()

# Aplicamos normalizacion
film_normalizado[normalizarFilm] = scaler.fit_transform(
    limpios["film"][normalizarFilm]
)

print("Ddtos despues de normalizar:")
print(film_normalizado[normalizarFilm].head())

comparacion = pd.DataFrame({
    "Media antes": limpios["film"][normalizarFilm].mean(),
    "Media despues": film_normalizado[normalizarFilm].mean(),
    "Desviacion antes": limpios["film"][normalizarFilm].std(ddof=0),
    "Desviacion despues": film_normalizado[normalizarFilm].std(ddof=0)
})

display(comparacion)


Ddtos despues de normalizar:
     length  rental_duration  rental_rate  replacement_cost
0 -0.724445         0.719374    -1.209308          0.166341
1 -1.664897        -1.406855     1.221461         -1.156452
2 -1.615399         1.428118     0.006077         -0.164357
3  0.042766         0.010631     0.006077          1.158437
4  0.364499         0.719374     0.006077          0.497040


,Media antes,Media despues,Desviacion antes,Desviacion despues
length,115.272,-1.421085e-16,40.406114,1.0
rental_duration,4.985,-2.522427e-16,1.410948,1.0
rental_rate,2.980,-3.819167e-16,1.645570,1.0
replacement_cost,19.984,-4.192202e-16,6.047807,1.0
